# Project 06: Marketing Attribution & Campaign Performance
Beginner-friendly notebook. Just run each cell from top to bottom (Shift + Enter).

**Data:** We create a *simulated* marketing dataset so you don't need to download anything.
(Simulated data is clearly labelled - in a real project you would load a public campaign CSV instead.)

## Step 0: Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", "{:,.2f}".format)
print("Libraries ready!")

## Step 0b: Create the simulated data
Two tables:
1. `df` - daily campaign data per channel (spend, impressions, clicks, ...)
2. `journeys` - customer touchpoints (which channels a customer saw before buying)

You don't need to understand this cell. Just run it.

In [ ]:
rng = np.random.default_rng(42)          
dates = pd.date_range("2025-01-01", periods=90)

settings = {
    "Search":  (200, 20, 0.050, 0.90, 0.30, 0.012, 60),
    "Social":  (150, 8,  0.015, 0.80, 0.20, 0.015, 45),
    "Email":   (50,  20, 0.080, 0.95, 0.40, 0.020, 55),
    "Display": (120, 4,  0.005, 0.70, 0.10, 0.015, 40),
}


messy_names = {
    "Search":  ["Search", "search ", "SEARCH", "google search"],
    "Social":  ["Social", "social", "Social Media", "SOCIAL "],
    "Email":   ["Email", "email", "E-mail", "EMAIL "],
    "Display": ["Display", "display", "Display Ads", "DISPLAY "],
}

rows = []
for d in dates:
    for ch, (spend, cpm, ctr, sess, lead, cvr, aov) in settings.items():
        s = spend * rng.uniform(0.8, 1.2)
        imp = s / cpm * 1000
        clk = imp * ctr * rng.uniform(0.85, 1.15)
        ses = clk * sess
        lds = ses * lead
        conv = ses * cvr * rng.uniform(0.8, 1.2)
        rows.append({
            "date": d, "channel": rng.choice(messy_names[ch]),
            "spend": round(s, 2), "impressions": int(imp), "clicks": int(clk),
            "sessions": int(ses), "leads": int(lds),
            "conversions": int(conv), "orders": int(conv),
            "revenue": round(int(conv) * aov * rng.uniform(0.9, 1.1), 2),
        })

df = pd.DataFrame(rows)


df = pd.concat([df, df.sample(5, random_state=1)], ignore_index=True)


channel_list = list(settings.keys())
j = []
for cust in range(1, 801):
    n_touch = rng.integers(1, 4)                     
    path = rng.choice(channel_list, size=n_touch, p=[0.35, 0.25, 0.15, 0.25])
    value = round(rng.uniform(30, 90), 2)
    for step, ch in enumerate(path, start=1):
        j.append({"customer_id": cust, "step": step, "n_steps": n_touch,
                  "channel": ch, "order_value": value})
journeys = pd.DataFrame(j)

print("Campaign table:", df.shape, "| Journey table:", journeys.shape)
df.head()

## Step 1: Standardise channel names and validate the data grain
**Grain** = what one row means. Here: *one row = one channel on one day*. So (date, channel) should be unique.

In [ ]:
name_map = {
    "search": "Search", "google search": "Search",
    "social": "Social", "social media": "Social",
    "email": "Email", "e-mail": "Email",
    "display": "Display", "display ads": "Display",
}
df["channel"] = df["channel"].str.strip().str.lower().map(name_map)

print("Channels after cleaning:", df["channel"].unique())
print("Missing channel names:", df["channel"].isna().sum())

In [ ]:
dupes = df.duplicated(subset=["date", "channel"], keep="first")
print("Duplicate rows found:", dupes.sum())

df = df[~dupes].copy()          
print("Rows after removing duplicates:", len(df))
print("Expected rows (90 days x 4 channels):", 90 * 4)

## Step 2: Reconcile spend, clicks, sessions and conversions
The marketing funnel should only go **down**:
impressions >= clicks >= sessions >= conversions. If not, something is wrong with the data.

In [ ]:
problems = {
    "clicks > impressions":    (df["clicks"] > df["impressions"]).sum(),
    "sessions > clicks":       (df["sessions"] > df["clicks"]).sum(),
    "conversions > sessions":  (df["conversions"] > df["sessions"]).sum(),
    "negative spend":          (df["spend"] < 0).sum(),
    "missing values":          df.isna().sum().sum(),
}
for name, count in problems.items():
    print(f"{name}: {count}")

In [ ]:
df.groupby("channel")[["spend", "impressions", "clicks", "sessions", "conversions", "revenue"]].sum()

## Step 3: Calculate KPIs
| KPI | Meaning | Formula |
|---|---|---|
| CTR | Click-through rate | clicks / impressions |
| CPC | Cost per click | spend / clicks |
| CVR | Conversion rate | conversions / sessions |
| CPA | Cost per acquisition | spend / conversions |
| ROAS | Return on ad spend | revenue / spend |

In [ ]:
channel = df.groupby("channel")[["spend", "impressions", "clicks", "sessions",
                                 "conversions", "revenue"]].sum()

channel["CTR"]  = channel["clicks"] / channel["impressions"]
channel["CPC"]  = channel["spend"] / channel["clicks"]
channel["CVR"]  = channel["conversions"] / channel["sessions"]
channel["CPA"]  = channel["spend"] / channel["conversions"]
channel["ROAS"] = channel["revenue"] / channel["spend"]

channel[["spend", "revenue", "CTR", "CPC", "CVR", "CPA", "ROAS"]]

## Step 4: Use NumPy to measure uncertainty
A single ROAS number can be misleading. We use **bootstrapping**: resample the days many times
to see how much ROAS could vary. This gives a 95% confidence interval.

In [ ]:
n_boot = 2000
ci_rows = []

for ch in df["channel"].unique():
    d = df[df["channel"] == ch]
    spend = d["spend"].to_numpy()
    revenue = d["revenue"].to_numpy()

    boot = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(d), size=len(d))      
        boot.append(revenue[idx].sum() / spend[idx].sum())

    low, high = np.percentile(boot, [2.5, 97.5])
    ci_rows.append({"channel": ch, "ROAS": revenue.sum() / spend.sum(),
                    "CI_low": low, "CI_high": high})

ci = pd.DataFrame(ci_rows).set_index("channel")
ci

In [ ]:
best = ci["ROAS"].idxmax()
print("Highest ROAS channel:", best)
for ch in ci.index:
    if ch != best:
        overlap = ci.loc[ch, "CI_high"] >= ci.loc[best, "CI_low"]
        print(f"{ch} vs {best}: intervals overlap? {overlap}")

## Step 5: Time, audience and channel summaries in pandas

In [ ]:
df["month"] = df["date"].dt.to_period("M").astype(str)
df["weekday"] = df["date"].dt.day_name()

monthly = df.pivot_table(index="month", columns="channel", values="revenue", aggfunc="sum")
print("Revenue by month and channel:")
display(monthly)

weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
weekday = df.groupby("weekday")[["spend", "revenue"]].sum().reindex(weekday_order)
weekday["ROAS"] = weekday["revenue"] / weekday["spend"]
print("Performance by weekday:")
display(weekday)

In [ ]:
df.groupby("channel")["conversions"].agg(["mean", "std", "min", "max"])

## Step 6: Compare attribution rules
**Attribution** = deciding which channel gets credit for a sale.
- **Last-touch:** the final channel before the order gets 100%
- **First-touch:** the first channel gets 100%
- **Linear:** every channel in the journey gets equal credit

In [ ]:

first = journeys[journeys["step"] == 1].set_index("customer_id")["channel"]
last  = journeys[journeys["step"] == journeys["n_steps"]].set_index("customer_id")["channel"]
value = journeys.groupby("customer_id")["order_value"].first()

last_touch  = value.groupby(last).sum()
first_touch = value.groupby(first).sum()

journeys["linear_credit"] = journeys["order_value"] / journeys["n_steps"]
linear = journeys.groupby("channel")["linear_credit"].sum()

attribution = pd.DataFrame({"last_touch": last_touch,
                            "first_touch": first_touch,
                            "linear": linear})
attribution["last_vs_linear_diff"] = attribution["last_touch"] - attribution["linear"]
attribution

In [ ]:
attribution[["last_touch", "first_touch", "linear"]].plot(kind="bar", figsize=(8, 4))
plt.title("Revenue credit by attribution rule")
plt.ylabel("Revenue credit")
plt.xticks(rotation=0)
plt.show()

## Step 7: Budget reallocation scenario (with constraints)
**Rules and assumptions (explicit!):**
- Move at most **20%** of the lowest-ROAS channel's budget to the highest-ROAS channel
- Extra spend gets **diminishing returns**: it earns only **50%** of the channel's average ROAS
- Cut spend also loses only 50% of average ROAS (the weakest spend is usually the least useful)

Change the numbers below to test other scenarios.

In [ ]:
max_shift = 0.20         
marginal_factor = 0.50    

roas = channel["ROAS"]
spend_by_ch = channel["spend"]
worst, best = roas.idxmin(), roas.idxmax()

moved = spend_by_ch[worst] * max_shift
revenue_lost   = moved * roas[worst] * marginal_factor
revenue_gained = moved * roas[best]  * marginal_factor
net_change = revenue_gained - revenue_lost

print(f"Move {moved:,.0f} of budget from {worst} (ROAS {roas[worst]:.2f}) to {best} (ROAS {roas[best]:.2f})")
print(f"Revenue lost:   {revenue_lost:,.0f}")
print(f"Revenue gained: {revenue_gained:,.0f}")
print(f"Net revenue change: {net_change:,.0f}  ({net_change / channel['revenue'].sum():.2%} of total revenue)")

In [ ]:
results = []
for shift in [0.05, 0.10, 0.20, 0.30]:
    for factor in [0.3, 0.5, 0.8]:
        m = spend_by_ch[worst] * shift
        net = m * roas[best] * factor - m * roas[worst] * factor
        results.append({"shift": shift, "marginal_factor": factor, "net_revenue_change": net})

scenarios = pd.DataFrame(results).pivot(index="shift", columns="marginal_factor", values="net_revenue_change")
scenarios

## Channel dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

channel["spend"].plot(kind="bar", ax=axes[0, 0], title="Total spend")
channel["ROAS"].plot(kind="bar", ax=axes[0, 1], title="ROAS", color="green",
                     yerr=[channel["ROAS"] - ci["CI_low"], ci["CI_high"] - channel["ROAS"]], capsize=4)
channel["CPA"].plot(kind="bar", ax=axes[1, 0], title="Cost per acquisition (CPA)", color="orange")
monthly.plot(ax=axes[1, 1], title="Monthly revenue by channel", marker="o")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

## Step 8: Measurement caveats (write these in your report!)
1. **Attribution is not causation.** Last-touch credit does not prove a channel *caused* the sale.
2. **Last-touch bias:** it over-rewards channels near purchase (e.g. Search, Email) and ignores channels that created awareness (e.g. Display, Social).
3. **Overlap:** customers see several channels, so credits across channels double count if you add up each channel's own reported conversions.
4. **Incrementality:** some of these customers would have bought anyway. Only a controlled test (holdout group, geo test) can measure true lift.
5. **Simulated data:** results here come from made-up numbers, so findings only demonstrate the method.
6. **Budget scenario:** based on assumed diminishing returns, not measured ones. Real response curves need more data or experiments.

### Final submission checklist
- [x] Campaign KPI table (Step 3)
- [x] Attribution comparison (Step 6)
- [x] Budget scenario analysis (Step 7)
- [x] Channel dashboard (above)
- [x] Measurement caveats (Step 8)